# Transformer From Scratch

Practice Session by Amodh Herath with parallel to DataCamp Transfomers with Pytorch and Attention is all you need paper. Here a transformer architecture is built from scratch using pytorch for the learning and understanding of this architecture.

In [2]:
import torch
import math
import torch.nn as nn

f:\Projects\transformer\.transformerenv\Lib\site-packages\torch\_subclasses\functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


## Input Embedding

In [3]:


class InputEmbedding(nn.Module):
    def __init__(self,vocab_size:int , d_model:int) -> None:
        super().__init__()  
        self.d_model=d_model
        self.vocab_size=vocab_size
        self.embedding=nn.Embedding(num_embeddings=vocab_size,embedding_dim=d_model)  
    
    def forward(self,x):
        return self.embedding(x)*math.sqrt(self.d_model)

## Positional Encoding


Generate the positional Encoding for tokens based on its position by using sin and cosine values.
-   even (2i) --> Sin(pos/1000^(2i/d_model))
-   odd (2i+1) --> Cos(pos/1000^(2i/d_model))

In [ ]:
class PositionalEncoding(nn.Module):
    pe: torch.Tensor
    def __init__(self, d_model, max_seq_len) -> None:
        super().__init__()
        # Initial 1D tensor filled with zeroes
        pe=torch.zeros(size=(max_seq_len,d_model))

        # position of the token from (0,...max_sequence_length)
        position=torch.arange(start=0 , end=max_seq_len).unsqueeze(1)
        

        '''
        Calculate the divisional term (1000^(2i/d_model)), it was translated to exponential base for efficient computations.
        the denominators  are identical for every pair of adjacent indices (2i and 2i+1).there are 
        only d_model/2 unique frequencies so we can use step=2
        how this would look like --> 0,2,4,6,...d_model-2
        
        in case of 2i absolute last number in this sequence, we substitute the maximum possible value of 
        2i (which we established is 2*(d_model/2 -1 )
        2i --> 0,2,4,6,8...d_model-2
        '''
        
        div_term=torch.exp(torch.arange(start=0 , end=d_model , step=2, dtype=torch.float)*-math.log(10000)/d_model)
        
        # all rows and all even columns
        pe[:,0::2] = torch.sin(position*div_term)
        # all rows and all odd columns
        pe[:,1::2] = torch.cos(position*div_term)
        
        # add this as a non trainable paramter but part of model state
        self.register_buffer('pe',pe.unsqueeze(0))
        
    def forward(self,x):
        return x + self.pe[:,:x.size(0)]
        
        
        